author - ransell

# Data Loading (X & YouTube)



Filtering down to the data needed for the BT Ranking implementation
This does not include the data on the likert scale, or the open ended questions

Data needed: VALUES-humanness_march_2025_YT_X_March 13, 2025_11.13.csv

In [1]:
import pandas as pd

# Name of the ID column in main file
id_column = "ResponseId"

df = pd.read_csv("VALUES-humanness_march_2025_YT_X_March 13, 2025_11.13.csv")

columns_to_keep = [id_column]  # always keep the ID column

for col in df.columns:
    if col.startswith("x_res_") or col.startswith("yt_res_"):
        columns_to_keep.append(col)
    elif col.startswith("x_pairs_") or col.startswith("yt_pairs_"):
        columns_to_keep.append(col)


columns_to_keep = list(dict.fromkeys(columns_to_keep))

subset_df = df[columns_to_keep]

# 4. Read the CSV that contains failed-attention IDs
failed_attn_df = pd.read_csv("data/x_yt_failed_attn_ids.csv")
failed_ids = failed_attn_df[id_column].tolist()

# Filter out rows whose ResponseId is in the failed ID list
subset_df = subset_df[~subset_df[id_column].isin(failed_ids)]


subset_df.to_csv("data/x_yt_subset_data.csv", index=False)

print(subset_df.head())


          ResponseId yt_res_1 yt_res_2 yt_res_3 yt_res_4 yt_res_10 yt_res_5  \
2  R_1qetdPkPOVvDPtg        2        1        1        2         1        1   
4  R_7Gfa7GArBwEBclA        2        1        2        2         1        1   
5  R_32AirxeIiztE0sp        2        2        1        2         1        2   
6  R_57d42gWnCJawbw1        2        2        1        2         1        2   
7  R_12FllPszS1WdDym        1        1        1        2         1        1   

  yt_res_6 yt_res_7 yt_res_8  ... yt_pairs_1.id.0  \
2        2        2        2  ...             491   
4        2        1        1  ...             401   
5        2        2        1  ...              56   
6        2        2        1  ...             324   
7        1        2        1  ...             331   

                                   yt_pairs_1.post.0 yt_pairs_2.id.0  \
2  foreign policy is so complex... but I think @K...             341   
4          trump's character is a major concern... 😔          

Code block to get one pair per row

In [2]:
df = pd.read_csv("data/x_yt_subset_data.csv")

# List to hold the long-format rows
rows = []

# Loop over every row (participant) in the dataset
for idx, row in df.iterrows():
    participant_id = row["ResponseId"]  

    # Loop over all 'yt' questions 
    for i in range(1, 16):
        # Skip question #10 (attention check)
        if i == 10:
            continue

        res_col        = f"yt_res_{i}"
        left_id_col    = f"yt_pairs_{i}.left_id.0"
        left_text_col  = f"yt_pairs_{i}.left_text.0"
        right_text_col = f"yt_pairs_{i}.right_text.0"
        right_id_col   = f"yt_pairs_{i}.right_id.0"

        if res_col in df.columns:
            rows.append({
                "ResponseId":    participant_id,
                "question_type": "yt",
                "question_num":  i,
                "user_response": row.get(res_col),
                "left_id":       row.get(left_id_col),
                "left_text":     row.get(left_text_col),
                "right_text":    row.get(right_text_col),
                "right_id":      row.get(right_id_col)
            })

    # Loop over all 'x' questions 
    for i in range(1, 16):
        # Skip question #10 (attention check)
        if i == 10:
            continue

        res_col        = f"x_res_{i}"
        left_id_col    = f"x_pairs_{i}.left_id.0"
        left_text_col  = f"x_pairs_{i}.left_text.0"
        right_text_col = f"x_pairs_{i}.right_text.0"
        right_id_col   = f"x_pairs_{i}.right_id.0"

        if res_col in df.columns:
            rows.append({
                "ResponseId":    participant_id,
                "question_type": "x",
                "question_num":  i,
                "user_response": row.get(res_col),
                "left_id":       row.get(left_id_col),
                "left_text":     row.get(left_text_col),
                "right_text":    row.get(right_text_col),
                "right_id":      row.get(right_id_col)
            })

# Convert the list of dictionaries into a DataFrame 
reshaped_df = pd.DataFrame(rows)

# Function to assign winner and loser based on the user's response
def assign_winner(row):
    response = str(row['user_response']).strip()  # stripped of whitespace
    if response == '1':
        return pd.Series({
            "post_winner": row["left_text"],
            "id_winner": row["left_id"],
            "post_loser": row["right_text"],
            "id_loser": row["right_id"]
        })
    elif response == '2':
        return pd.Series({
            "post_winner": row["right_text"],
            "id_winner": row["right_id"],
            "post_loser": row["left_text"],
            "id_loser": row["left_id"]
        })
    else:
        # If the response doesn't match expected values, defaults to None
        return pd.Series({
            "post_winner": None,
            "id_winner": None,
            "post_loser": None,
            "id_loser": None
        })

# Apply the function to every row to create new columns for winners and losers
winner_loser_cols = reshaped_df.apply(assign_winner, axis=1)
reshaped_df = pd.concat([reshaped_df, winner_loser_cols], axis=1)

output_csv = "data/x_yt_reshaped_data_with_winner.csv"
reshaped_df.to_csv(output_csv, index=False)

print("Reshaped data with winner/loser information saved to:", output_csv)
print(reshaped_df.head())


Reshaped data with winner/loser information saved to: data/x_yt_reshaped_data_with_winner.csv
          ResponseId question_type  question_num  user_response  left_id  \
0  R_1qetdPkPOVvDPtg            yt             1              2      316   
1  R_1qetdPkPOVvDPtg            yt             2              1      552   
2  R_1qetdPkPOVvDPtg            yt             3              1      382   
3  R_1qetdPkPOVvDPtg            yt             4              2      545   
4  R_1qetdPkPOVvDPtg            yt             5              1      451   

                                           left_text  \
0         can someone please fact check this guy?? 😂   
1  Kamala was on fire tonight! 🔥 She really took ...   
2  Is it just me or is Kamala Harris's voice a bi...   
3  Trump just keeps repeating the same old lies. ...   
4  Kamala's calmness was refreshing, but seriousl...   

                                          right_text  right_id  \
0       '@USER I am a cat and dog eater with k

We now have the data seperated into X and YT, along with the winners and losers and their IDs.

In [3]:
# Filter for yt and x question types and select only the required columns
yt_df = reshaped_df.loc[reshaped_df["question_type"] == "yt", 
                          ["ResponseId", "post_winner", "id_winner", "post_loser", "id_loser"]]
x_df = reshaped_df.loc[reshaped_df["question_type"] == "x", 
                         ["ResponseId", "post_winner", "id_winner", "post_loser", "id_loser"]]

# Save the two DataFrames to separate CSV files
yt_df.to_csv("data/yt_data.csv", index=False)
x_df.to_csv("data/x_data.csv", index=False)

print("Saved yt questions to 'yt_data.csv' and x questions to 'x_data.csv'.")

Saved yt questions to 'yt_data.csv' and x questions to 'x_data.csv'.
